**2. MOCO TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri job
job_name = "prova"                      # nome da passare poi a notebook moco 
dataset = "Test"                        # Test, Standard, Anomalies*
weights_encoder_sar = "Test"     
weights_encoder_opt = ""                
#handler = pretrain_encoders            # time_debug = True solo per debug, = False per training

parametri = {
    "epochs": 1, 
    "batch_size": 16,
    "lr": 0.03,
    "weight_decay": 1e-4,
    "momentum": 0.9,
    "patch_size": 256,
    "n_images1": 4, "n_channels1": 2,           # sar
    "n_images2": 4, "n_channels2": 10,          # ottico
    "moco_dim": 128,
    "moco_k": 1554,                               # dimensioni coda (multiplo batch_size), da modificare 24867/16 = 1554     "moco_m": 0.999,
    "moco_t": 0.07,
    "symmetric": False,
    "mamba": False,
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "weights_encoder_sar": weights_encoder_sar,
    "weights_encoder_opt": weights_encoder_opt,
    "patience": 20,
    "min_delta": 1e-4    
}

print(f"PARAMETRI: {parametri}")

# volume
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]



**BUILD ENVIRONMENT**

In [ ]:
moco_func = project.new_function(
    name= f'moco_Floods_{dataset}-{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_moco", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)



# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = moco_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_moco = moco_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",
    # local_execution= True,                       # 1x = 1 gpu
    wait=True
)

print(f"Run moco avviato: {run_moco.id}")
print(run_moco.status.state)
print(run_moco.status.message)

**PLOTS**

In [ ]:
# salvataggio log
path_moco = project.get_artifact(f"moco-metrics_{dataset}_{job_name}").download(overwrite=True)
df_moco = pd.read_csv(path_moco)

# plot
plt.figure(figsize=(8, 5))
plt.plot(df_moco['epoch'], df_moco['train_loss'], color='green', label='Train Loss')
plt.title(f'Training MoCo - {dataset}_{job_name}')
plt.xlabel('Epochs')
plt.ylabel('Loss (InfoNCE)')
plt.grid(True)
plt.legend()
plt.show()